# tides_spin_migration — migration-driven obliquity tides

Two Earth-sized planets migrating through a gas disk while tides evolve their spins. Exercises two REBOUNDx forces simultaneously and a parameter changed in the middle of the run (migration is switched off half-way). Verified bit-identical to the C.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd reboundx_rust
cargo build --release --example tides_spin_migration
cd porttest
../target/release/examples/tides_spin_migration 62.83185307179586
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "reboundx_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "tides_spin_migration"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.00s


In [2]:
import struct

def unbits(h):
    """Turn a 16-hex-digit IEEE-754 bit pattern back into a float."""
    return struct.unpack("<d", int(h, 16).to_bytes(8, "little"))[0]

def read_state(path):
    """Read one of the raw-bit state dumps into {label: [floats]}."""
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if not parts:
                continue
            key, rest = parts[0], parts[1:]
            vals = []
            for tok in rest:
                if len(tok) == 16:
                    try:
                        vals.append(unbits(tok))
                        continue
                    except ValueError:
                        pass
                vals.append(tok)
            out.setdefault(key, []).append(vals)
    return out

def compare(a, b, label_a="C", label_b="Rust"):
    """Byte-compare two dump files and report."""
    ta = open(a, "rb").read().replace(b"\r\n", b"\n")
    tb = open(b, "rb").read().replace(b"\r\n", b"\n")
    if ta == tb:
        print(f"BIT-IDENTICAL: {label_a} and {label_b} agree on every bit")
        return True
    print(f"MISMATCH between {label_a} and {label_b}")
    la, lb = ta.decode().splitlines(), tb.decode().splitlines()
    for i, (x, y) in enumerate(zip(la, lb)):
        if x != y:
            print(f"  line {i}:\n    {label_a}: {x}\n    {label_b}: {y}")
    return False


In [3]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + EXE)
res = subprocess.run([exe, "62.83185307179586"], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

migration done t=6.28318530717958623e1



In [4]:
name = EXAMPLE.replace("tides_spin_", "")
rust = os.path.join(WORK, f"state_{name}_rust.txt")
cref = os.path.join(WORK, f"state_{name}_c.txt")
if os.path.exists(rust):
    st = read_state(rust)
    print("--- final state (decoded from raw bits) ---")
    for p in st.get("p", []):
        print(f"  particle {p[0]}: x={p[1]:+.9e} y={p[2]:+.9e} z={p[3]:+.9e}")
    for o in st.get("Omega", []):
        if len(o) > 1 and not isinstance(o[1], str):
            mag = (o[1]**2 + o[2]**2 + o[3]**2) ** 0.5
            print(f"  spin {o[0]}: |Omega| = {mag:.9e}")
if os.path.exists(cref):
    print()
    compare(cref, rust)
else:
    print("\n(no C reference dump present - build and run the C harness to compare)")


--- final state (decoded from raw bits) ---
  particle 0: x=-5.328717988e-06 y=+2.582376088e-06 z=+7.070755521e-09
  particle 1: x=+1.338008725e-01 y=-1.076717991e-01 z=-8.991584685e-04
  particle 2: x=+2.214469934e-01 y=-6.448660671e-02 z=+4.277747671e-04
  spin 0: |Omega| = 1.825000000e+01
  spin 1: |Omega| = 7.299961466e+01
  spin 2: |Omega| = 1.216664737e+02

MISMATCH between C and Rust
  line 0:
    C: example tides_spin_migration_driven_obliquity_tides tmax 4083a28c59d5433b
    Rust: example tides_spin_migration_driven_obliquity_tides tmax 404f6a7a2955385e
  line 1:
    C: t 4083a28c59d5433b
    Rust: t 404f6a7a2955385e
  line 4:
    C: p 0 3ea965028d223b02 3ec21c94a77f8926 be5509834e78de07 bef7e75ccd42f93d bedcc390dcf8a85a bea03dbef942daa7 3ff0000000000000
    Rust: p 0 bed659aad8e5a2bb 3ec5a99c448e3489 3e3e5e60bee43d38 bf006840c401b981 bf0eb970e207f842 be5bd57965d2c5ce 3ff0000000000000
  line 5:
    C: Omega 0 3e8699fa9179b368 3ebaea2fde0989d0 40323fffffffffeb
    Rust: Omega 0